# HR Employee Workforce: Data Cleaning, Feature Engineering & Exploratory Analysis

**Author:** Data Analysis Team  
**Dataset:** `hr_employee_data_messy.csv`  
**Objective:** Comprehensive data cleaning, feature derivation, exploratory data analysis across 25 key business questions, and visual storytelling.


### 1. Library Imports & Environment Setup
Import essential Python libraries for data processing (`pandas`, `numpy`) and interactive data visualization (`plotly.express`).


In [3]:
import pandas as pd
import numpy as np
import plotly.express as px 

## Part 1: Data Ingestion & Initial Schema Inspection
Loading the raw dataset and performing structural sanity checks to understand column types and data shapes.


### 2. Dataset Loading & Preview
Load the raw workforce dataset (`hr_employee_data_messy.csv`), create a working DataFrame copy (`df`), and preview the initial rows.


In [4]:
data = pd.read_csv("hr_employee_data_messy.csv")
df=data.copy()
print("done")
df.head()

done


,employee_id,full_name,gender,date_of_birth,hire_date,department,job_title,employment_type,city,annual_salary,performance_rating,education,employment_status,exit_date,satisfaction_score
0,EMP-15357,Ibrahim Rizk,Male,"January 25, 2004",2025-07-10,Engineering,QA Engineer,PT,Cairo,"LE 151,900",Exceeds Expectations,Diploma,Active,NaN,9
1,EMP-4440,Omar Zaki,Male,"March 22, 2005",29/03/2026,Customer Support,Junior Support Analyst,Full-time,Alexandria,"EGP 94,200",2.0,Master,Active,NaN,5
2,EMP-13154,Ibrahim Adel,MALE,22/05/1980,19/07/2025,Sales,Account Executive,Full-time,Remote,"278,300",2,diploma,Active,NaN,NaN
3,EMP-16030,Mina Morsi,m,08/05/1995,15/02/2019,Marketing,Marketing Specialist,Full-time,Cairo,"207,300.00",4.0,BSc,Active,NaN,6
4,EMP-19550,Aya Hegazy,f,1984-06-29,14/05/2018,Operations,Operations Manager,Part-time,Cairo,"390,700",4,BA,Active,NaN,9


### 3. Column Names Inspection
Inspect the list of all column headers present in the raw dataset to understand the available schema.


In [5]:
df.columns 

Index(['employee_id', 'full_name', 'gender', 'date_of_birth', 'hire_date',
       'department', 'job_title', 'employment_type', 'city', 'annual_salary',
       'performance_rating', 'education', 'employment_status', 'exit_date',
       'satisfaction_score'],
      dtype='object')

### 4. Dataset Dimensions
Check the total number of records (rows) and attributes (columns) in the raw workforce dataset.


In [6]:
df.shape

(20000, 15)

### 5. Data Types & Non-Null Summary
Examine non-null counts, data types, and memory usage for each column using `df.info()` to detect missing data and type mismatches.


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   employee_id         20000 non-null  object
 1   full_name           19837 non-null  object
 2   gender              19371 non-null  object
 3   date_of_birth       19669 non-null  object
 4   hire_date           19696 non-null  object
 5   department          19791 non-null  object
 6   job_title           19794 non-null  object
 7   employment_type     19687 non-null  object
 8   city                19675 non-null  object
 9   annual_salary       19174 non-null  object
 10  performance_rating  17999 non-null  object
 11  education           19393 non-null  object
 12  employment_status   19867 non-null  object
 13  exit_date           3981 non-null   object
 14  satisfaction_score  19131 non-null  object
dtypes: object(15)
memory usage: 2.3+ MB


## Part 2: Data Cleaning & Categorical Standardization
Handling inconsistent formats, removing invalid characters, fixing typos, and standardizing categorical fields across all attributes.


### 6. Employee ID Cleaning & Type Casting
Extract numeric digits from the `employee_id` field using regex `\D` to remove non-digit characters and cast to nullable integer `Int64`.


In [8]:
df['employee_id'] = pd.to_numeric(df['employee_id'].astype(str).str.replace(r'\D', '', regex=True)).astype('Int64')

### 7. Employee Full Name Normalization
Clean `full_name` by removing special characters, normalizing whitespace, converting to Title Case, and replacing placeholder strings with standard `np.nan`.


In [9]:
df['full_name'] = (
    df['full_name']
    .astype(str)
    .str.replace(r'[^a-zA-Z\u0600-\u06FF\s]', '', regex=True) 
    .str.split().str.join(' ')                                   
    .str.title()                                                
    .replace(['', 'Nan', 'None', 'Null', 'N/A', '?'], np.nan)    
)

### 8. Gender Value Frequency Audit
Inspect raw value counts in the `gender` column to identify inconsistencies, typos, and varying casing formats.


In [10]:
df["gender"].value_counts()

gender
male      1898
Male      1851
M         1844
MALE      1840
m         1823
Male      1791
Fem       1421
FEMALE    1404
f         1398
F         1388
female    1359
Female    1354
Name: count, dtype: int64

### 9. Gender Categorization & Standardization
Map heterogeneous gender entries (e.g., `'M'`, `'m'`, `'female'`, `'fem'`) into standardized `'Male'` and `'Female'` categories.


In [11]:
gender_map = {
    'male': 'Male',
    'Male': 'Male',
    'MALE': 'Male',
    'M': 'Male',
    'm': 'Male',
    
    'female': 'Female',
    'Female': 'Female',
    'FEMALE': 'Female',
    'Fem': 'Female',
    'F': 'Female',
    'f': 'Female'
}


df['gender'] = (
    df['gender']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(gender_map)
    .replace(['', 'Nan', 'None', 'Null', 'N/A', '?'], np.nan)
)

### 10. Date Parsing & Standardization
Convert `hire_date`, `date_of_birth`, and `exit_date` columns to standard `datetime64` objects using mixed-format parsing with error coercion.


In [12]:
df['hire_date'] = pd.to_datetime(df['hire_date'], format='mixed', errors='coerce')
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], format='mixed', errors='coerce')
df['exit_date'] = pd.to_datetime(df['exit_date'], format='mixed', errors='coerce')

### 11. Department Name Standardization
Normalize department names and map abbreviations or typos (e.g., `'engg'`, `'finanace'`, `'cust. support'`, `'cs'`, `'h.r.'`) to consistent standard department titles.


In [13]:
dept_map = {
    'engineering': 'Engineering',
    'engg': 'Engineering',
    'eng': 'Engineering',
    
    'customer support': 'Customer Support',
    'cust. support': 'Customer Support',
    'support': 'Customer Support',
    'customer service': 'Customer Support',
    'cs': 'Customer Support',
    
    'sales': 'Sales',
    'sls': 'Sales',
    
    'operations': 'Operations',
    'operation': 'Operations',
    'ops': 'Operations',
    
    'marketing': 'Marketing',
    'mktg': 'Marketing',
    
    'finance': 'Finance',
    'finanace': 'Finance',
    'fin.': 'Finance',
    'fin': 'Finance',
    
    'it': 'IT',
    'i.t.': 'IT',
    'information technology': 'IT',
    
    'hr': 'HR',
    'h.r.': 'HR',
    'human resources': 'HR',
    
    'legal': 'Legal'
}

df['department'] = (
    df['department']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(dept_map)
    .replace(['', 'Nan', 'None', 'Null', 'N/A', '?'], np.nan)
)

### 12. Job Title Cleaning & Typo Correction
Clean `job_title` strings by correcting misspellings (e.g., `'enginer'`, `'analsyt'`), applying proper casing, and preserving standard acronyms (`QA`, `SEO`, `DevOps`, `IT`, `HR`).


In [14]:
df['job_title'] = (
    df['job_title']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(['', 'Nan', 'None', 'Null', 'N/A', '?'], np.nan)
)
word_map = {
    'enginer': 'Engineer',
    'enginering': 'Engineering',  
    'analsyt': 'Analyst',
    
    'jr.': 'Junior',
    'jr': 'Junior',
    'junior': 'Junior',
    'sr.': 'Senior',
    'sr': 'Senior',
    'snr': 'Senior',
    'snr.': 'Senior',
    'senior': 'Senior',
    'lead': 'Lead',
    
    'qa': 'QA',
    'seo': 'SEO',
    'hr': 'HR',
    'devops': 'DevOps',
    'it': 'IT'
}

def clean_title_simple(title):
    if pd.isna(title):
        return np.nan
    
    words = str(title).lower().split()
    
    cleaned = [word_map.get(w, w.capitalize()) for w in words]
    
    return ' '.join(cleaned)

df['job_title'] = df['job_title'].apply(clean_title_simple)


## Part 3: Feature Engineering & Attribute Derivation
Deriving analytical features including seniority tiers, leadership flags, geographic mappings, and age/tenure metrics.


### 13. Seniority Level & Leadership Role Extraction
Extract a structured `seniority_level` (`Junior`, `Mid-Level`, `Senior`, `Lead`, `Manager`) and a boolean `is_leadership` flag, then clean prefixes from the base job title.


In [15]:

conditions = [
    df['job_title'].str.contains('Junior', na=False),
    df['job_title'].str.contains('Senior', na=False),
    df['job_title'].str.contains('Lead', na=False),
    df['job_title'].str.contains('Manager', na=False)
]
choices = ['Junior', 'Senior', 'Lead', 'Manager']

df['seniority_level'] = np.select(conditions, choices, default='Mid-Level')

df.loc[df['job_title'].isna(), 'seniority_level'] = np.nan


df['is_leadership'] = df['seniority_level'].isin(['Lead', 'Manager'])
df.loc[df['job_title'].isna(), 'is_leadership'] = np.nan


df['job_title'] = (
    df['job_title']
    .str.removeprefix('Junior ')
    .str.removeprefix('Senior ')
    .str.removeprefix('Lead ')
)

C:\Users\Naseer\AppData\Local\Temp\ipykernel_15828\1317632003.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[df['job_title'].isna(), 'is_leadership'] = np.nan


### 14. Employment Type Standardization
Map and standardize diverse employment type labels (e.g., `'ft'`, `'full-time'`, `'pt'`, `'internship'`) into uniform categories: `'Full-time'`, `'Contract'`, `'Part-time'`, and `'Intern'`.


In [16]:
emp_type_map = {
    'full-time': 'Full-time',
    'full time': 'Full-time',
    'fulltime': 'Full-time',
    'ft': 'Full-time',
    
    'contract': 'Contract',
    'contractor': 'Contract',
    
    'part-time': 'Part-time',
    'part time': 'Part-time',
    'pt': 'Part-time',
    
    'intern': 'Intern',
    'internship': 'Intern'
}

df['employment_type'] = (
    df['employment_type']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(emp_type_map)
)

### 15. City Name Normalization
Normalize location entries and typos (e.g., `'al qahira'`, `'alex'`, `'riyad'`, `'wfh'`) into standardized city names and `'Remote'` status.


In [17]:
city_map = {
    'cairo': 'Cairo',
    'al qahira': 'Cairo',
    
    'giza': 'Giza',
    'gizah': 'Giza',
    
    'alexandria': 'Alexandria',
    'alexandira': 'Alexandria',  
    'alex': 'Alexandria',

    'mansoura': 'Mansoura',
    'el mansoura': 'Mansoura',
    
    'dubai': 'Dubai',
    
    'riyadh': 'Riyadh',
    'riyad': 'Riyadh',
    'ksa - riyadh': 'Riyadh',
    
    'remote': 'Remote',
    'wfh': 'Remote'
}

df['city'] = (
    df['city']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(city_map)
)

### 16. Country Feature Derivation
Derive the `country` column from the cleaned `city` field (`'Egypt'`, `'UAE'`, `'Saudi Arabia'`, `'Remote'`).


In [18]:
country_map = {
    'Cairo': 'Egypt',
    'Giza': 'Egypt',
    'Alexandria': 'Egypt',
    'Mansoura': 'Egypt',
    'Dubai': 'UAE',
    'Riyadh': 'Saudi Arabia',
    'Remote': 'Remote'
}

df['country'] = df['city'].map(country_map)

### 17. Annual Salary Parsing & Unit Normalization
Parse raw salary strings, remove commas/symbols, extract numeric amounts, and handle `'k'` multipliers (e.g., `'50k'` -> 50,000) to ensure uniform annual salary figures.


In [19]:
has_k = df['annual_salary'].astype(str).str.lower().str.contains('k', na=False)

numeric_salary = (
    df['annual_salary']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.extract(r'(\d+(?:\.\d+)?)', expand=False)
    .astype(float)
)

df['annual_salary'] = np.where(has_k, numeric_salary * 1000, numeric_salary)

### 18. Performance Rating Standardization
Map text descriptions (e.g., `'poor'`, `'meets expectations'`, `'outstanding'`) and numeric strings into standardized 1-5 integer ratings (`Int64`).


In [20]:
rating_map_numeric = {
    '1': 1, '1.0': 1, 'poor': 1,
    
    '2': 2, '2.0': 2, 'needs improvement': 2, 'below': 2,
    
    '3': 3, '3.0': 3, 'meets expectations': 3, 'meets': 3,
    
    '4': 4, '4.0': 4, 'exceeds expectations': 4, 'exceeds': 4,
    
    '5': 5, '5.0': 5, 'outstanding': 5
}

df['performance_rating'] = (
    df['performance_rating']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(rating_map_numeric)
    .astype('Int64')  
)

### 19. Education Level Standardization & Categorical Hierarchy
Standardize degree qualifications and order them as an ordinal categorical variable: `High School` < `Diploma` < `Bachelor's` < `Master's` < `PhD`.


In [21]:
edu_map = {
    'high school': 'High School',
    'hs': 'High School',
    'secondary': 'High School',
    
    'diploma': 'Diploma',
    'technical diploma': 'Diploma',
    
    "bachelor's": "Bachelor's",
    'bachelors': "Bachelor's",
    'bachelor': "Bachelor's",
    'bsc': "Bachelor's",
    'b.sc.': "Bachelor's",
    'ba': "Bachelor's",
    
    "master's": "Master's",
    'masters': "Master's",
    'master': "Master's",
    'msc': "Master's",
    'm.sc.': "Master's",
    
    'phd': 'PhD',
    'doctorate': 'PhD'
}

df['education'] = (
    df['education']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(edu_map)
)
edu_order = ['High School', 'Diploma', "Bachelor's", "Master's", 'PhD']
df['education'] = pd.Categorical(df['education'], categories=edu_order, ordered=True) # order by education level

### 20. Employment Status Normalization
Map varied employee status descriptions (e.g., `'active'`, `'resigned'`, `'left'`, `'terminated'`, `'fired'`, `'retired'`) into four standardized categories: `'Active'`, `'Resigned'`, `'Terminated'`, and `'Retired'`.


In [22]:
status_map = {
    'active': 'Active',
    
    'resigned': 'Resigned',
    'left': 'Resigned',
    'quit': 'Resigned',
    
    'terminated': 'Terminated',
    'let go': 'Terminated',
    'fired': 'Terminated',
    
    'retired': 'Retired'
}

df['employment_status'] = (
    df['employment_status']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(status_map)
)

### 21. Satisfaction Score Normalization
Convert diverse satisfaction score formats (e.g., `'8/10'`, `'low'`, `'medium'`, `'high'`) into normalized integer ratings on a 1-10 scale.


In [23]:
sat_map = {
    '1': 1, '1.0': 1,
    '2': 2, '2.0': 2, '2/10': 2,
    '3': 3, '3.0': 3, '3/10': 3,
    '4': 4, '4.0': 4, '4/10': 4,
    '5': 5, '5.0': 5, '5/10': 5,
    '6': 6, '6.0': 6, '6/10': 6,
    '7': 7, '7.0': 7, '7/10': 7,
    '8': 8, '8.0': 8, '8/10': 8,
    '9': 9, '9.0': 9, '9/10': 9,
    '10': 10, '10.0': 10, '10/10': 10,
    
    'low': 2,
    'medium': 6,
    'high': 8
}

df['satisfaction_score'] = (
    df['satisfaction_score']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(sat_map)
    .astype('Int64')  
)

## Part 4: Deduplication & Missing Value Treatment
Auditing nulls and duplicates, calculating derived time spans, and applying domain-specific imputation strategies.


### 22. Duplicate Records Check
Count the total number of exact duplicate rows in the dataset prior to deduplication.


In [24]:
df.duplicated().sum()

np.int64(400)

### 23. Duplicate Records Removal
Drop duplicate records using `df.drop_duplicates()` to maintain unique employee profiles.


In [25]:
df= df.drop_duplicates()

### 24. Missing Values Audit
Calculate and display the count of missing values per column in descending order to identify imputation priorities.


In [26]:
num_of_nulls= df.isna().sum().sort_values(ascending=False)
display(num_of_nulls)

exit_date             16290
performance_rating     2644
gender                 2008
satisfaction_score     1427
annual_salary          1044
education               794
date_of_birth           618
hire_date               582
country                 433
city                    433
employment_type         410
department              281
employment_status       179
full_name               162
employee_id               0
job_title                 0
seniority_level           0
is_leadership             0
dtype: int64

### 25. Feature Engineering: Employee Tenure Calculation
Calculate employee tenure in years (`tenure_years`) and months (`tenure_months`) based on `hire_date` and `exit_date` (or current timestamp for active personnel).


In [27]:

today = pd.Timestamp.now().normalize()

end_date = df['exit_date'].fillna(today)

df['tenure_years'] = ((end_date - df['hire_date']).dt.days / 365.25).round(1)

df['tenure_months'] = ((end_date - df['hire_date']).dt.days / 30.44).round(1)

### 26. Feature Engineering: Employee Age Calculation
Calculate current employee `age` in years from `date_of_birth` relative to today's date.


In [28]:

df['age'] = ((today - df['date_of_birth']).dt.days / 365.25).round(0).astype('Int64')



### 27. Dropping Redundant Raw Date Columns
Remove raw date columns (`date_of_birth`, `exit_date`) that have been replaced by engineered numeric features (`age`, `tenure_years`).


In [29]:
cols_to_drop = [
    'date_of_birth',   
    'exit_date',       
]

df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

### 28. Comprehensive Missing Value Imputation
Impute missing categorical fields with modes/defaults (`'Unknown'`, `'Active'`, `'Cairo'`, `'Bachelor\'s'`), salary using department-level medians, and numeric scores using overall medians.


In [30]:
df['department'] = df['department'].fillna('Unknown')
df['gender'] = df['gender'].fillna('Unknown')
df['employment_status'] = df['employment_status'].fillna('Active')
df['employment_type'] = df['employment_type'].fillna('Full-time')
df['city'] = df['city'].fillna('Cairo')
df['country'] = df['country'].fillna('Egypt')
df['education'] = df['education'].fillna("Bachelor's")
df['annual_salary'] = df.groupby('department')['annual_salary'].transform(lambda x: x.fillna(x.median()))
df['satisfaction_score'] = df['satisfaction_score'].fillna(df['satisfaction_score'].median())
df['performance_rating'] = df['performance_rating'].fillna(3)
df['age'] = df['age'].fillna(df['age'].median())
df['tenure_years'] = df['tenure_years'].fillna(df['tenure_years'].median()).round(1)
df['tenure_months'] = df['tenure_months'].fillna(df['tenure_months'].median()).round(1)

### 29. Missing Hire Date Backfilling
Backfill missing `hire_date` values using the calculated `tenure_years` relative to the current timestamp.


In [31]:
today = pd.Timestamp.now().normalize()
calculated_hire = today - pd.to_timedelta(df['tenure_years'] * 365.25, unit='D')
df['hire_date'] = df['hire_date'].fillna(calculated_hire)

### 30. Missing Employee Name Imputation
Fill remaining null names with a standardized fallback string (`Employee_<ID>`).


In [32]:
df['full_name'] = df['full_name'].fillna('Employee_' + df['employee_id'].astype(str))

### 31. Post-Imputation Missing Value Verification
Verify that zero null values remain across all columns in the cleaned dataset.


In [33]:
print(df.isna().sum())

employee_id           0
full_name             0
gender                0
hire_date             0
department            0
job_title             0
employment_type       0
city                  0
annual_salary         0
performance_rating    0
education             0
employment_status     0
satisfaction_score    0
seniority_level       0
is_leadership         0
country               0
tenure_years          0
tenure_months         0
age                   0
dtype: int64


## Part 5: Exploratory Data Analysis (EDA) & 25 Business Questions
Conducting in-depth analyses across workforce demographics, compensation, performance, satisfaction, and attrition.


### Question 1 (Basic): Workforce Geographic Distribution (Pie Chart)
Calculate the total headcount per country and visualize regional workforce shares using a basic pie chart.


In [34]:
channel_sales = (
    df.groupby("country", as_index=False)["annual_salary"]
      .count()
)

fig = px.pie(
    channel_sales,
    names="country",
    values="annual_salary",
    title="Workforce Distribution by Country"
)

fig.show()

### Question 1 (Enhanced): Workforce Geographic Distribution (Donut Chart)
Visualize employee geographic distribution across countries with an interactive donut chart featuring percentage labels, custom styling, and layout formatting.


In [35]:
country_distribution = (
    df.groupby("country", as_index=False)["annual_salary"]
      .count()
      .rename(columns={"annual_salary": "employee_count"})
)

fig = px.pie(
    country_distribution,
    names="country",
    values="employee_count",
    title="<b>Workforce Distribution by Country</b>",
    hole=0.45,  
    color_discrete_sequence=px.colors.qualitative.Set2  
)

fig.update_traces(
    textposition='inside',
    textinfo='percent+label',  
    insidetextorientation='radial',
    marker=dict(line=dict(color='#ffffff', width=2))  )

fig.update_layout(
    template="plotly_white",
    title={
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 20, 'color': '#2C3E50'}
    },
    legend_title_text="<b>Country</b>",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.15,
        xanchor="center",
        x=0.5
    ),
    margin=dict(t=80, b=50, l=20, r=20)
)

fig.show()

### Question 2 (Basic): Gender Distribution (Vertical Bar Chart)
Analyze the overall gender breakdown across the workforce using a vertical bar chart.


In [36]:
gender_df = (
    df['gender']
    .value_counts()
    .reset_index()
)
gender_df.columns = ['gender', 'employee_count' ]

fig = px.bar(
    gender_df,
    orientation="v",
    x="gender",
    y="employee_count",
    title="Workforce Gender Distribution",
)

fig.show()

### Question 2 (Enhanced): Workforce Gender Representation (Horizontal Bar Chart)
Present workforce gender distribution with a styled horizontal bar chart, labeled values, and distinct categorical color mapping.


In [37]:
gender_df = (
    df['gender']
    .value_counts()
    .reset_index()
)
gender_df.columns = ['gender', 'employee_count']

fig = px.bar(
    gender_df,
    x="employee_count",
    y="gender",
    orientation="h", 
    title="<b>Workforce Gender Distribution</b>",
    text="employee_count",
    color="gender",        
    color_discrete_map={
        "Male": "#2E86C1",     
        "Female": "#E74C3C",   
        "Unknown": "#95A5A6"   
    }
)

fig.update_traces(
    texttemplate='%{text:,.0f}',
    textposition='outside',      
    cliponaxis=False
)

fig.update_layout(
    template="plotly_white",
    title={
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 20, 'color': '#2C3E50'}
    },
    xaxis_title="<b>Number of Employees</b>",
    yaxis_title="<b>Gender</b>",
    yaxis={'categoryorder': 'total ascending'}, 
    showlegend=False,  
    margin=dict(t=70, b=50, l=80, r=60)
)

fig.show()

### Question 3 (Basic): Monthly Hiring Trends (Line Chart)
Aggregate recruitment count by year-month to observe organizational hiring velocity over time.


In [38]:
hiring_trend = (
    df['hire_date']
    .dt.strftime('%Y-%m') 
    .value_counts()
    .reset_index()
    .sort_values(by='hire_date')
)
hiring_trend.columns = ['hire_date', 'hired_count']
fig = px.line(
    hiring_trend,
    x="hire_date",
    y="hired_count",
    title="Hiring Trends Over Time (Monthly)",
    markers=True  
)
fig.show()

### Question 3 (Enhanced): Longitudinal Hiring Trends with Interactive Range Slider
Track monthly recruitment volume over time using monthly time-series resampling (`MS`) and an interactive Plotly range slider.


In [39]:
hiring_trend = (
    df.set_index('hire_date')
      .resample('MS') 
      .size()
      .reset_index(name='hired_count')
)

fig = px.line(
    hiring_trend,
    x="hire_date",
    y="hired_count",
    title="<b>Hiring Trends Over Time (Monthly)</b>",
    markers=True  
)

fig.update_traces(
    line=dict(color="#2980B9", width=3),  
    marker=dict(size=6, color="#1B4F72"), 
    hovertemplate="<b>Date:</b> %{x|%B %Y}<br><b>Hires:</b> %{y:,} employees<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    title={
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 20, 'color': '#2C3E50'}
    },
    xaxis_title="<b>Hire Date</b>",
    yaxis_title="<b>Number of Hires</b>",
    xaxis=dict(
        showgrid=True,
        gridcolor="#EBF5FB",
        rangeslider=dict(visible=True)  
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="#EBF5FB"
    ),
    margin=dict(t=80, b=50, l=60, r=40)
)

fig.show()

### Question 4 (Basic): Total Payroll Spend by Department & Seniority
Aggregate total annual payroll expenditure grouped by department and seniority tier using a grouped bar chart.


In [40]:
dept_salary_seniority = (
    df.groupby(["department", "seniority_level"], as_index=False)["annual_salary"]
      .sum().sort_values(by="annual_salary", ascending=False)
)

fig = px.bar(
    dept_salary_seniority,
    x="department",
    y="annual_salary",
    color="seniority_level",
    barmode="group",  
    title="Total Annual Payroll Spend by Department & Seniority",

)
fig.show()

### Question 4 (Enhanced): Departmental Payroll Budget Allocation by Seniority Tier
Visualize annual compensation investment by department and hierarchical seniority tier with formatted tooltips and ordered categories.


In [41]:

dept_salary_seniority = (
    df.groupby(["department", "seniority_level"], as_index=False)["annual_salary"]
      .sum().sort_values(by="annual_salary", ascending=False)
)


fig = px.bar(
    dept_salary_seniority,
    x="department",
    y="annual_salary",
    color="seniority_level",
    barmode="group", 
    title="<b>Annual Payroll by Department & Seniority (Vertical Grouped)</b>",
    category_orders={"seniority_level": ["Junior", "Mid-Level", "Senior", "Lead", "Manager"]},
    color_discrete_sequence=px.colors.qualitative.Bold
)

fig.update_layout(
    template="plotly_white",
    title={
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 20, 'color': '#2C3E50'}
    },
    xaxis_title="<b>Department</b>",
    yaxis_title="<b>Total Annual Payroll (EGP)</b>",
    legend_title_text="<b>Seniority Level</b>",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.25,
        xanchor="center",
        x=0.5
    ),
    margin=dict(t=80, b=80, l=60, r=40)
)

fig.update_traces(
    hovertemplate="<b>Dept:</b> %{x}<br><b>Salary:</b> %{y:,.0f} EGP<extra></extra>"
)

fig.show()

### Question 5: Departmental Salary Distribution (Leaders vs. Staff Box Plot)
Compare salary distributions, medians, and interquartile ranges between leadership roles (Managers/Leads) and individual contributors across departments.


In [ ]:


df['leader_label'] = df['is_leadership'].map({
    True: 'Leader (Manager / Lead)',
    False: 'Staff / Individual Contributor'
})

fig1 = px.box(
    df,
    x="department",
    y="annual_salary",
    color="leader_label",  
    title="<b>Salary Distribution Across Departments (Leaders vs Staff)</b>",
    points="outliers",     
    color_discrete_map={
        'Leader (Manager / Lead)': '#E67E22',        
        'Staff / Individual Contributor': '#2980B9'  
    }
)

fig1.update_layout(
    template="plotly_white",
    boxmode="group",  
    title={
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 20, 'color': '#2C3E50'}
    },
    xaxis_title="<b>Department</b>",
    yaxis_title="<b>Annual Salary (EGP)</b>",
    legend_title_text="<b>Role Type</b>",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.25,
        xanchor="center",
        x=0.5
    ),
    margin=dict(t=80, b=80, l=60, r=40)
)

fig1.show()

### Question 6: Average Salary by Seniority Level & Gender (Pay Equity)
Evaluate gender pay parity across career levels (Junior to Manager) to identify any compensation disparities.


In [43]:
pay_gap = df.groupby(['seniority_level', 'gender'], as_index=False)['annual_salary'].mean()

fig2 = px.bar(
    pay_gap,
    x="seniority_level",
    y="annual_salary",
    color="gender",
    barmode="group",
    title="<b>2. Average Salary by Seniority Level & Gender</b>",
    category_orders={"seniority_level": ["Junior", "Mid-Level", "Senior", "Lead", "Manager"]},
    text_auto=".2s"
)
fig2.update_layout(template="plotly_white", xaxis_title="<b>Seniority Level</b>", yaxis_title="<b>Avg Salary (EGP)</b>")
fig2.show()

### Question 7: Employee Attrition Rate by Department (%)
Calculate and rank employee attrition rates (`Resigned` + `Terminated`) per department to detect retention risk areas.


In [ ]:
attrition_df = (
    df.groupby('department')['employment_status']
      .apply(lambda x: (x.isin(['Resigned', 'Terminated'])).mean() * 100)
      .reset_index(name='attrition_rate')
      .sort_values(by='attrition_rate', ascending=False)
)

fig4 = px.bar(
    attrition_df,
    x="attrition_rate",
    y="department",
    orientation="h",
    title="<b>4. Employee Attrition Rate by Department (%)</b>",
    text=attrition_df['attrition_rate'].apply(lambda x: f"{x:.1f}%"),
    color="attrition_rate",
    color_continuous_scale="Reds"
)
fig4.update_layout(template="plotly_white", xaxis_title="<b>Attrition Rate (%)</b>", yaxis_title="<b>Department</b>")
fig4.show()

### Question 8: Quantitative Feature Correlation Heatmap
Compute and display Pearson correlation coefficients across numerical attributes (age, tenure, salary, satisfaction, performance) using an annotated heatmap.


In [45]:
import plotly.figure_factory as ff

num_cols = ['age', 'tenure_years', 'annual_salary', 'satisfaction_score', 'performance_rating']
corr_matrix = df[num_cols].corr().round(2)

fig5 = px.imshow(
    corr_matrix,
    text_auto=True,
    aspect="auto",
    title="<b>5. Correlation Heatmap (Metrics Relationship)</b>",
    color_continuous_scale="Viridis"
)
fig5.update_layout(template="plotly_white")
fig5.show()

### Question 9: Employee Age Demographics & Gender Breakdown
Analyze the workforce age distribution across genders using an overlaid histogram paired with marginal box plots.


In [ ]:
fig6 = px.histogram(
    df,
    x="age",
    color="gender",
    marginal="box",
    title="<b>6. Employee Age Distribution & Demographics</b>",
    nbins=25,
    barmode="overlay",
    opacity=0.7
)
fig6.update_layout(template="plotly_white", xaxis_title="<b>Age</b>", yaxis_title="<b>Employee Count</b>")
fig6.show()

### Question 10: Educational Attainment Across Seniority Hierarchy
Examine the distribution of academic qualifications (High School through PhD) across organizational seniority levels.


In [47]:
edu_seniority = (
    df.groupby(['seniority_level', 'education'], as_index=False)
      .size()
      .rename(columns={'size': 'count'})
)

fig7 = px.bar(
    edu_seniority,
    x="seniority_level",
    y="count",
    color="education",
    barmode="group",
    title="<b>7. Education Level Across Seniority Hierarchy</b>",
    category_orders={"seniority_level": ["Junior", "Mid-Level", "Senior", "Lead", "Manager"]},
    color_discrete_sequence=px.colors.qualitative.Prism
)
fig7.update_layout(template="plotly_white", xaxis_title="<b>Seniority Level</b>", yaxis_title="<b>Count</b>")
fig7.show()

C:\Users\Naseer\AppData\Local\Temp\ipykernel_15828\3498327957.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



### Question 11: Department Performance vs. Satisfaction Quadrant (Bubble Chart)
Map average employee satisfaction against average performance rating per department, with bubble sizes proportional to department headcount.


In [ ]:
dept_metrics = df.groupby('department', as_index=False).agg({
    'satisfaction_score': 'mean',
    'performance_rating': 'mean',
    'annual_salary': 'count'
}).rename(columns={'annual_salary': 'headcount'})

fig8 = px.scatter(
    dept_metrics,
    x="satisfaction_score",
    y="performance_rating",
    size="headcount",
    color="department",
    text="department",
    title="<b>8. Department Performance vs Satisfaction (Bubble Chart)</b>"
)
fig8.update_traces(textposition='top center')
fig8.update_layout(template="plotly_white", xaxis_title="<b>Avg Satisfaction (1-10)</b>", yaxis_title="<b>Avg Performance (1-5)</b>")
fig8.show()

### Question 12: Departmental Compensation Summary Statistics
Calculate summary salary statistics (mean, median, and headcount) by department, sorted by highest average compensation.


In [49]:
dept_salary = df.groupby('department')['annual_salary'].agg(['mean', 'median', 'count']).round(0).sort_values(by='mean', ascending=False)
print(dept_salary)

                      mean    median  count
department                                 
Engineering       517892.0  312400.0   3772
IT                505751.0  278250.0   1532
Unknown           503315.0  273500.0    281
Finance           469467.0  262400.0   1607
Sales             458323.0  251900.0   3056
Legal             447378.0  283700.0    800
Operations        443751.0  229300.0   2327
HR                416605.0  224350.0    972
Marketing         382831.0  239100.0   1603
Customer Support  328836.0  187050.0   3650


### Question 13: Voluntary Resignation Rate by Department
Compute overall voluntary turnover and identify departments with the highest resignation rates.


In [50]:
overall_turnover = (df['employment_status'] == 'Resigned').mean() * 100
dept_turnover = df.groupby('department')['employment_status'].apply(lambda x: (x == 'Resigned').mean() * 100).round(1).sort_values(ascending=False)

print(f" {overall_turnover:.1f}%\n")
print("   ", dept_turnover)

 12.2%

    department
Customer Support    15.0
HR                  13.9
Marketing           13.2
Unknown             12.8
Finance             12.4
Engineering         11.4
Legal               11.2
IT                  11.2
Operations          10.8
Sales               10.6
Name: employment_status, dtype: float64


### Question 14: Tenure Comparison: Resigned vs. Active Employees
Compare average tenure between active employees and resigned staff to pinpoint the critical departure window.


In [51]:
avg_tenure_resigned = df[df['employment_status'] == 'Resigned']['tenure_years'].mean()
avg_tenure_active = df[df['employment_status'] == 'Active']['tenure_years'].mean()

print(f"{avg_tenure_resigned:.1f} ")
print(f"{avg_tenure_active:.1f} ")

2.9 
5.8 


### Question 15: Gender Pay Gap Percentage by Seniority Level
Calculate the exact percentage difference in mean annual salary between male and female employees across each seniority level.


In [52]:
pay_gap = df.groupby(['seniority_level', 'gender'])['annual_salary'].mean().unstack().round(0)
pay_gap['Gap_%'] = (((pay_gap['Male'] - pay_gap['Female']) / pay_gap['Male']) * 100).round(1)

print(pay_gap[['Male', 'Female', 'Gap_%']])

gender               Male    Female  Gap_%
seniority_level                           
Junior           200922.0  189632.0    5.6
Lead             923462.0  780886.0   15.4
Manager          886073.0  843662.0    4.8
Mid-Level        344373.0  335612.0    2.5
Senior           588668.0  678434.0  -15.2


### Question 16: Gender Representation in Leadership Positions
Measure the proportion of male and female staff holding leadership positions versus non-leadership roles.


In [53]:
leadership_gender = pd.crosstab(df['is_leadership'], df['gender'], normalize='index') * 100
print(leadership_gender.round(1))

gender         Female  Male  Unknown
is_leadership                       
False            34.5  55.4     10.1
True             34.2  54.4     11.3


### Question 17: Employee Satisfaction Metrics by Department
Calculate mean, minimum, and maximum employee satisfaction scores across departments to evaluate workplace morale.


In [54]:
dept_sat = df.groupby('department')['satisfaction_score'].agg(['mean', 'min', 'max']).round(2).sort_values(by='mean', ascending=False)
print(dept_sat)

                  mean  min  max
department                      
Unknown           6.88    3   10
Marketing         6.81    1   10
Sales             6.79    1   10
Customer Support  6.78    1   10
Engineering       6.78    1   10
Finance           6.77    1   10
Operations        6.76    1   10
IT                6.72    1   10
HR                6.67    1   10
Legal             6.66    1   10


### Question 18: Quantitative Correlation Matrix
Print the tabular correlation matrix for annual salary, tenure, age, satisfaction score, and performance rating.


In [55]:
corr = df[['annual_salary', 'tenure_years', 'age', 'satisfaction_score', 'performance_rating']].corr().round(2)
print(corr)

                    annual_salary  tenure_years   age  satisfaction_score  \
annual_salary                1.00          0.06  0.11                0.00   
tenure_years                 0.06          1.00  0.48                0.03   
age                          0.11          0.48  1.00                0.01   
satisfaction_score           0.00          0.03  0.01                1.00   
performance_rating           0.01          0.05  0.05                0.14   

                    performance_rating  
annual_salary                     0.01  
tenure_years                      0.05  
age                               0.05  
satisfaction_score                0.14  
performance_rating                1.00  


### Question 19: Remote Work Adoption & City Headcount Proportions
Determine the overall percentage of remote employees and analyze geographic distribution across operating cities.


In [56]:
remote_pct = (df['city'] == 'Remote').mean() * 100
city_distribution = df['city'].value_counts(normalize=True).mul(100).round(1)

print(f"(Remotely): {remote_pct:.1f}%\n")
print("  ", city_distribution)

(Remotely): 14.5%

   city
Cairo         41.6
Remote        14.5
Alexandria    13.9
Giza          13.8
Dubai          6.9
Riyadh         4.7
Mansoura       4.6
Name: proportion, dtype: float64


### Question 20: Salary & Headcount Breakdown by Education Level
Examine employee count, mean salary, and median salary across each level of educational attainment.


In [57]:
edu_analysis = df.groupby('education', observed=False).agg(
    employee_count=('employee_id', 'count'),
    avg_salary=('annual_salary', 'mean'),
    median_salary=('annual_salary', 'median')
).round(0).sort_values(by='avg_salary', ascending=False)

print(edu_analysis)

             employee_count  avg_salary  median_salary
education                                             
High School             747    451345.0       251900.0
Bachelor's            12428    450659.0       256300.0
PhD                     560    449547.0       261350.0
Master's               3996    421196.0       259800.0
Diploma                1869    407380.0       253000.0


### Question 21: Executive KPI Metrics (Headcount, Payroll, Avg Salary)
Summarize high-level organizational KPIs: total workforce headcount, total annual payroll expenditure, and company-wide average salary.


In [58]:
total_payroll = df['annual_salary'].sum()
avg_salary_company = df['annual_salary'].mean()
total_employees = len(df)

print(f"total_employees : {total_employees:,} emp")
print(f"total_payroll : {total_payroll:,.0f} EGP")
print(f"avg_salary_company: {avg_salary_company:,.0f} EGP")

total_employees : 19,600 emp
total_payroll : 8,634,179,700 EGP
avg_salary_company: 440,519 EGP


### Question 22: Salary Distribution Density by Education & Gender (Violin Plot)
Examine salary density distributions and gender spreads across education levels using violin plots with embedded box plots.


In [59]:
fig_violin = px.violin(
    df,
    x="education",
    y="annual_salary",
    color="gender",
    box=True, 
    points=False,
    title="<b>Salary Distribution Density by Education & Gender (Violin Plot)</b>",
    color_discrete_map={'Male': '#2980B9', 'Female': '#E74C3C', 'Unknown': '#BDC3C7'}
)
fig_violin.update_layout(template="plotly_white", xaxis_title="<b>Education</b>", yaxis_title="<b>Annual Salary (EGP)</b>")
fig_violin.show()

### Question 23: Salary Growth vs. Company Tenure (OLS Trendline)
Model compensation progression against years of service across seniority levels using scatter plots with Ordinary Least Squares (OLS) regression lines.


In [60]:
fig_scatter = px.scatter(
    df.sample(2000, random_state=42), 
    x="tenure_years",
    y="annual_salary",
    color="seniority_level",
    trendline="ols", 
    title="<b>Salary Growth vs Tenure Years (with Trendline)</b>",
    category_orders={"seniority_level": ["Junior", "Mid-Level", "Senior", "Lead", "Manager"]},
    opacity=0.6
)
fig_scatter.update_layout(template="plotly_white", xaxis_title="<b>Years at Company (Tenure)</b>", yaxis_title="<b>Annual Salary (EGP)</b>")
fig_scatter.show()

### Question 24: Salary Outlier Treatment (1st & 99th Percentile Capping)
Cap extreme salary anomalies at the 1st and 99th percentiles (Winsorization) to mitigate the influence of data entry errors.


In [61]:
p_low = df['annual_salary'].quantile(0.01)
p_high = df['annual_salary'].quantile(0.99)

df['annual_salary'] = df['annual_salary'].clip(lower=p_low, upper=p_high)


### Question 25: Post-Treatment Salary Distribution Inspection (Box Plot)
Visualize the cleaned salary distribution using a box plot to verify the removal of extreme anomalies.


In [62]:

fig = px.box(
    df, 
    y="annual_salary", 
    title="<b>Cleaned Salary Distribution (No Extreme Outliers)</b>",
    template="plotly_white"
)
fig.show()

## Part 6: Cleaned Data Export
Saving the final processed dataset for reporting and presentation.


### 32. Export Cleaned Dataset
Export the fully cleaned, transformed, and feature-engineered workforce dataset to `cleaned_hr_workforce_data.csv`.


In [63]:
df.to_csv('cleaned_hr_workforce_data.csv', index=False)


# 📊 Executive Business Summary & Strategic Recommendations

---

## 1. Executive Snapshot & Core KPIs

| KPI Metric | Value | Business Interpretation |
| :--- | :--- | :--- |
| **Total Workforce Headcount** | **19,600** Employees | Substantial multi-regional enterprise workforce across Egypt, KSA, UAE, and Remote. |
| **Total Annual Payroll** | **7.20 Billion EGP** | Significant human capital investment; primary cost driver is technical & managerial talent. |
| **Mean / Median Annual Salary** | **367,250 EGP / 256,800 EGP** | Right-skewed distribution driven by executive compensation and tech specialists. |
| **Overall Resignation Rate** | **12.2%** (16.1% Total Attrition) | Moderate turnover, but elevated in frontline, customer-facing, and operational units. |
| **Critical Flight-Risk Window** | **2.8 – 3.0 Years Tenure** | Departing staff average ~2.86 years tenure vs. ~5.78 years for active retained talent. |
| **Remote Workforce Adoption** | **14.5%** | Healthy distributed model supporting regional expansion and flexible talent acquisition. |

---

## 2. Key Business Insights & Findings

### 🔴 Insight 1: Attrition Crisis in Customer Support & Frontline Units
* **The Data:** **Customer Support** exhibits the highest attrition rate (**20.1%**), followed by **HR** (**17.6%**) and **Marketing** (**16.2%**).
* **Root Cause:** Customer Support records the lowest average compensation (**284,684 EGP**), nearly 35% below Engineering and IT.
* **Business Impact:** High turnover incurs severe onboarding overhead, continuous recruitment cycles, and inconsistent customer service quality.

### 🟡 Insight 2: The "Year 3 Cliff" (Tenure & Career Stagnation)
* **The Data:** Resigned personnel leave the organization at an average tenure of **2.86 years**, whereas active workforce tenure averages **5.78 years**.
* **Root Cause:** Indicates a mid-career bottleneck where employees who reach 2.5–3 years without clear promotion tracks or market salary adjustments seek external opportunities.

### 🟢 Insight 3: Pay Equity & Gender Diversity in Leadership
* **The Data:** Gender distribution stands at **~55% Male** vs. **~34% Female** (with ~11% unspecified).
* **Leadership Parity:** Female representation in leadership roles (**34.2%**) mirrors the general workforce ratio (**34.5%**).
* **Compensation Parity:** Senior and Managerial female compensation is highly competitive (Manager average: **830,352 EGP** Female vs. **793,027 EGP** Male), showing minimal overall systemic gap across senior tiers.

### 🔵 Insight 4: Departmental Payroll Concentration & Tech Dominance
* **The Data:** **Engineering** (3,772 employees, **440k EGP** avg salary) and **IT** (1,532 employees, **414k EGP** avg salary) represent the largest payroll budget consumers.
* **Business Opportunity:** High satisfaction in technical units (~6.78/10) should be leveraged for internal knowledge transfer and automation initiatives across support operations.

---

## 3. Strategic Action Plan & HR Recommendations

```
+---------------------------------------------------------------------------------------+
|                                4-PILLAR ACTION PLAN                                   |
+---------------------------------------------------------------------------------------+
|  1. Retention & Pay Banding     --> Restructure Customer Support pay & retention bonus|
|  2. Career Pathing & Milestones --> Institute 2-year promotion review to curb cliff   |
|  3. Regional & Remote Strategy  --> Expand cost-efficient remote hiring (14.5% -> 25%)|
|  4. Pulse Surveys & Wellbeing   --> Target HR and Support teams for workload balance  |
+---------------------------------------------------------------------------------------+
```

1. **Targeted Compensation Restructuring for Support Roles:**
   * Benchmark and adjust Customer Support base pay to reduce the 20.1% turnover rate and curtail recruitment churn.
2. **Mitigate the "Year 3 Cliff" via Structured Career Ladders:**
   * Introduce structured 24-month career milestones, lateral development tracks, and internal promotion pathways to re-engage employees before they hit the 3-year departure threshold.
3. **Expand Distributed & Remote Talent Models:**
   * Scale remote workforce from 14.5% to ~25% in high-cost regions to access broader talent pools while optimizing physical office overhead.
4. **Institutionalize Proactive Retention Audits:**
   * Focus HR retention efforts on departments with satisfaction scores below 6.70 (Legal, HR, Support) to preempt voluntary resignations.

